# Tipos relacionados a datas

O Pydantic oferece vários tipos relacionados a datas.

Vejamos alguns.

> 📝 **Nota de revisão (reprodutibilidade):** a demonstração do `PastDatetime` depende de o fuso **local** ser diferente de UTC (o autor
> está no Arizona, UTC−7, sem horário de verão). Para o notebook dar o mesmo resultado em qualquer máquina (inclusive servidores em UTC),
> fixamos o fuso local do processo em `America/Phoenix` antes de tudo. (`time.tzset()` só existe em Unix; no Windows, ajuste o fuso do sistema.)

In [1]:
import os, time

os.environ["TZ"] = "America/Phoenix"
time.tzset()
print(f"time.tzname: {time.tzname!r}")

time.tzname: ('MST', 'MST')


In [2]:
from pydantic import (
    BaseModel,
    ConfigDict,
    PastDatetime,
    PastDate,
    AwareDatetime,
    NaiveDatetime,
    ValidationError,
)

`PastDatetime` restringe o campo a um datetime válido **e** no passado em relação à hora local atual.

A documentação não é muito clara nisso: como datetimes *naive* (sem fuso) são interpretados para decidir se estão no passado?

Eles são tratados como hora **local**, não UTC.

Vamos testar...

In [3]:
from datetime import datetime, timedelta

import pytz

local_one_hour_ago = datetime.now() - timedelta(hours=1)
print(f"local_one_hour_ago: {local_one_hour_ago!r}")

local_one_hour_ago: datetime.datetime(2026, 11, 11, 3, 47, 11, 723681)


No fuso do autor (AZ, sem horário de verão), esse horário em UTC seria:

In [4]:
utc_one_hour_ago = local_one_hour_ago.astimezone(pytz.utc)

Mas queremos esse datetime *naive*, não *aware*, para ver como o Pydantic o trata em relação à hora local.

In [5]:
utc_naive_one_hour_ago = utc_one_hour_ago.replace(tzinfo=None)
print(f"utc_naive_one_hour_ago: {utc_naive_one_hour_ago!r}")

utc_naive_one_hour_ago: datetime.datetime(2026, 11, 11, 10, 47, 11, 723681)


Agora, usando `PastDatetime`:

In [6]:
class Model(BaseModel):
    dt: PastDatetime

In [7]:
m = Model(dt=local_one_hour_ago)
print(f"m: {m!r}")

m: Model(dt=datetime.datetime(2026, 11, 11, 3, 47, 11, 723681))


In [8]:
m = Model(dt=utc_one_hour_ago)
print(f"m: {m!r}")

m: Model(dt=datetime.datetime(2026, 11, 11, 10, 47, 11, 723681, tzinfo=<UTC>))


Os dois funcionam.

E para ver que o Pydantic interpreta um datetime *naive* como hora local (o horário UTC "sem fuso" está 7 horas **à frente** da hora local,
logo no futuro):

In [9]:
try:
    Model(dt=utc_naive_one_hour_ago)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
dt
  Input should be in the past [type=datetime_past, input_value=datetime.datetime(2026, 1... 11, 10, 47, 11, 723681), input_type=datetime]
    For further information visit https://errors.pydantic.dev/2.13/v/datetime_past


Vejamos outros tipos de datetime.

`NaiveDatetime` exige que o datetime recebido seja *naive*.

In [10]:
class Model(BaseModel):
    dt: NaiveDatetime

In [11]:
print(f"local_one_hour_ago: {local_one_hour_ago!r}")

local_one_hour_ago: datetime.datetime(2026, 11, 11, 3, 47, 11, 723681)


In [12]:
print(f"Model(dt=local_one_hour_ago): {Model(dt=local_one_hour_ago)!r}")

Model(dt=local_one_hour_ago): Model(dt=datetime.datetime(2026, 11, 11, 3, 47, 11, 723681))


Mas, se passarmos um datetime *aware*:

In [13]:
print(f"utc_one_hour_ago: {utc_one_hour_ago!r}")

utc_one_hour_ago: datetime.datetime(2026, 11, 11, 10, 47, 11, 723681, tzinfo=<UTC>)


In [14]:
try:
    Model(dt=utc_one_hour_ago)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
dt
  Input should not have timezone info [type=timezone_naive, input_value=datetime.datetime(2026, 1...1, 723681, tzinfo=<UTC>), input_type=datetime]
    For further information visit https://errors.pydantic.dev/2.13/v/timezone_naive


E o oposto: exigir que o datetime seja *aware*:

In [15]:
class Model(BaseModel):
    dt: AwareDatetime

In [16]:
print(f"Model(dt=utc_one_hour_ago): {Model(dt=utc_one_hour_ago)!r}")

Model(dt=utc_one_hour_ago): Model(dt=datetime.datetime(2026, 11, 11, 10, 47, 11, 723681, tzinfo=<UTC>))


A validação falha se passarmos um *naive*:

In [17]:
try:
    Model(dt=local_one_hour_ago)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
dt
  Input should have timezone info [type=timezone_aware, input_value=datetime.datetime(2026, 11, 11, 3, 47, 11, 723681), input_type=datetime]
    For further information visit https://errors.pydantic.dev/2.13/v/timezone_aware


Como aritmética de datetime **não** é exatamente fácil, é comum acabarmos escrevendo validadores e serializadores customizados.

Por exemplo, a forma recomendada de lidar com datetimes nas aplicações Python é converter tudo para UTC (muitas vezes como *naive*).
Assim a aritmética fica muito mais simples, já que UTC não tem horário de verão. E só convertemos para um datetime localizado (normalmente
*aware*) ao "exibir" os dados para o usuário.

Para isso, podemos escrever um validador customizado que desserializa qualquer datetime para UTC, com alguma suposição sobre o fuso de um
datetime *naive*, e conversão de fuso quando ele é *aware*. Voltaremos a isso depois.

> 💡 **Complemento:** `PastDate` (importado acima) faz o mesmo para `date`; também existem `FutureDate` e `FutureDatetime`.

In [18]:
from datetime import date

class Birth(BaseModel):
    day: PastDate

try:
    Birth(day=date.today() + timedelta(days=1))
except ValidationError as ex:
    print(f'ex.errors()[0]["type"]: {ex.errors()[0]["type"]}')

ex.errors()[0]["type"]: date_past
